In [ ]:

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from patsy import dmatrix
from statsmodels.stats.outliers_influence import variance_inflation_factor

from src.config import (
    CLEANED_DATA_PATH,
    RESULTS_DIR,
    LOGISTIC_MODEL_BASIC_PATH,
    LOGISTIC_MODEL_COUNTRY_PATH,
    LOGISTIC_MODEL_TIME_ADJUSTED_PATH,
    LOGISTIC_MODEL_INTERACTION_PATH,
    LOGISTIC_ODDS_RATIOS_PATH,
    LOGISTIC_MODEL_COMPARISON_PATH,
    LOGISTIC_VIF_RESULTS_PATH,
    LOGISTIC_VIF_INTERACTION_PATH
)

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import statsmodels.formula.api as smf

from src.config import (
    CLEANED_DATA_PATH,
    RESULTS_DIR
)

print("Project root:", PROJECT_ROOT)
print("Cleaned data path:", CLEANED_DATA_PATH)
print("Results directory:", RESULTS_DIR)

Current working directory: d:\2.DAP391m_Project\notebooks
Project root: d:\2.DAP391m_Project
Project root exists: True
Project root: d:\2.DAP391m_Project
Cleaned data path: D:\2.DAP391m_Project\data\processed\dataset_cleaned.csv
Results directory: D:\2.DAP391m_Project\data\results


In [3]:
df = pd.read_csv(CLEANED_DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

display(df.head())

Dataset loaded successfully.
Shape: (288541, 9)

Missing values:
user_id            0
timestamp          0
group              0
landing_page       0
converted          0
country            0
time_raw           0
elapsed_minutes    0
time_bucket        0
dtype: int64


,user_id,timestamp,group,landing_page,converted,country,time_raw,elapsed_minutes,time_bucket
0,851104,11:48.6,control,old_page,0,US,11:48.6,11.810000,0-15_min
1,804228,01:45.2,control,old_page,0,US,01:45.2,1.753333,0-15_min
2,661590,55:06.2,treatment,new_page,0,US,55:06.2,55.103333,45-60_min
3,853541,28:03.1,treatment,new_page,0,US,28:03.1,28.051667,15-30_min
4,864975,52:26.2,control,old_page,1,US,52:26.2,52.436667,45-60_min


## Prepare Modeling Variables

Logistic Regression requires numerical variables for treatment assignment.

We create a treatment indicator:

- `treat = 0` for the control group / old_page
- `treat = 1` for the treatment group / new_page

Since the dataset was cleaned earlier, the control group corresponds to `old_page` and the treatment group corresponds to `new_page`.

In [4]:
df["treat"] = (df["group"] == "treatment").astype(int)

df[["group", "landing_page", "treat", "converted", "country"]].head()

,group,landing_page,treat,converted,country
0,control,old_page,0,0,US
1,control,old_page,0,0,US
2,treatment,new_page,1,0,US
3,treatment,new_page,1,0,US
4,control,old_page,0,1,US


In [5]:
pd.crosstab(df["group"], df["landing_page"])

landing_page,new_page,old_page
group,,
control,0,144226
treatment,144315,0


In [6]:
df["treat"].value_counts()

treat
1    144315
0    144226
Name: count, dtype: int64

## Balance Check after Cleaning

Before fitting Logistic Regression, we check whether the control and treatment groups remain balanced after preprocessing.

This is important because mismatch filtering may slightly change the original random assignment balance.

We check:

- number of users in control and treatment groups
- country distribution by treatment group
- time_bucket distribution by treatment group if available

In [7]:
# Count users by treatment group
treatment_balance = df.groupby("treat").agg(
    users=("user_id", "count"),
    conversions=("converted", "sum"),
    conversion_rate=("converted", "mean")
).reset_index()

treatment_balance["conversion_rate_percent"] = treatment_balance["conversion_rate"] * 100

treatment_balance

,treat,users,conversions,conversion_rate,conversion_rate_percent
0,0,144226,17349,0.120290,12.029038
1,1,144315,17134,0.118726,11.872640


In [8]:
country_balance = pd.crosstab(
    df["country"],
    df["treat"],
    normalize="columns"
) * 100

country_balance

treat,0,1
country,,
CA,4.949177,5.027890
UK,25.030161,24.849115
US,70.020662,70.122995


In [9]:
if "time_bucket" in df.columns:
    time_balance = pd.crosstab(
        df["time_bucket"],
        df["treat"],
        normalize="columns"
    ) * 100

    display(time_balance)
else:
    print("time_bucket column is not available.")

treat,0,1
time_bucket,,
0-15_min,24.772926,24.942660
15-30_min,25.135551,24.953747
30-45_min,24.996880,25.097876
45-60_min,25.094643,25.005717


## Separation Check

Logistic Regression can have problems if one subgroup has a conversion rate of 0% or 100%. This is called complete separation.

We check conversion rates by country and treatment group. If all groups have conversion rates between 0 and 1, there is no complete separation problem.

In [10]:
separation_check = df.groupby(["country", "treat"]).agg(
    users=("user_id", "count"),
    conversions=("converted", "sum"),
    conversion_rate=("converted", "mean")
).reset_index()

separation_check["conversion_rate_percent"] = separation_check["conversion_rate"] * 100

separation_check

,country,treat,users,conversions,conversion_rate,conversion_rate_percent
0,CA,0,7138,848,0.118801,11.880078
1,CA,1,7256,811,0.111770,11.176957
2,UK,0,36100,4330,0.119945,11.994460
3,UK,1,35861,4341,0.121051,12.105072
4,US,0,100988,12171,0.120519,12.051927
5,US,1,101198,11982,0.118402,11.840155


In [11]:
separation_problem = separation_check[
    (separation_check["conversion_rate"] == 0) |
    (separation_check["conversion_rate"] == 1)
]

if separation_problem.empty:
    print("No complete separation detected.")
else:
    print("Potential separation problem detected:")
    display(separation_problem)

No complete separation detected.


## Model 1: Baseline Treatment Model

The first Logistic Regression model includes only the treatment indicator.

Model:

converted ~ treat

This model is expected to be consistent with the previous two-proportion z-test because it only compares the treatment group against the control group.

It is included as a baseline reference model.

In [12]:
model_basic = smf.logit(
    "converted ~ treat",
    data=df
).fit()

model_basic.summary()

Optimization terminated successfully.
         Current function value: 0.365941
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:              converted   No. Observations:               288541
Model:                          Logit   Df Residuals:                   288539
Method:                           MLE   Df Model:                            1
Date:                Sun, 17 May 2026   Pseudo R-squ.:               7.940e-06
Time:                        22:05:24   Log-Likelihood:            -1.0559e+05
converged:                       True   LL-Null:                   -1.0559e+05
Covariance Type:            nonrobust   LLR p-value:                    0.1953
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.9897      0.008   -245.805      0.000      -2.006      -1.974
treat         -0.0149      0.011     -1.295      0.195      -0.037       0.008
==============================================================================
"""

## Model 2: Country-Adjusted Logistic Regression

The second model adjusts for country-level baseline differences.

Model:

converted ~ treat + C(country)

This checks whether the treatment effect remains after controlling for country.

In [13]:
model_country = smf.logit(
    "converted ~ treat + C(country)",
    data=df
).fit()

model_country.summary()

Optimization terminated successfully.
         Current function value: 0.365935
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:              converted   No. Observations:               288541
Model:                          Logit   Df Residuals:                   288537
Method:                           MLE   Df Model:                            3
Date:                Sun, 17 May 2026   Pseudo R-squ.:               2.289e-05
Time:                        22:05:26   Log-Likelihood:            -1.0559e+05
converged:                       True   LL-Null:                   -1.0559e+05
Covariance Type:            nonrobust   LLR p-value:                    0.1843
====================================================================================
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -2.0307      0.027    -75.975      0.000      -2.083      -1.978
C(country)[T.UK]     0.0503      0.029      1.765      0.078      -0.006       0.106
C(country)[T.US]     0.0405      0.027      1.502      0.133      -0.012       0.093
treat               -0.0148      0.011     -1.291      0.197      -0.037       0.008
====================================================================================
"""

## Model 2b: Country + Time-Bucket Adjusted Logistic Regression

This model adds `time_bucket` as an additional covariate.

Model:

converted ~ treat + C(country) + C(time_bucket)

This serves as a robustness check to see whether the treatment effect remains stable after accounting for time-related variation.

In [14]:
if "time_bucket" in df.columns:
    model_time_adjusted = smf.logit(
        "converted ~ treat + C(country) + C(time_bucket)",
        data=df
    ).fit()

    display(model_time_adjusted.summary())
else:
    print("time_bucket column is not available. Model 2b is skipped.")

Optimization terminated successfully.
         Current function value: 0.365927
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:              converted   No. Observations:               288541
Model:                          Logit   Df Residuals:                   288534
Method:                           MLE   Df Model:                            6
Date:                Sun, 17 May 2026   Pseudo R-squ.:               4.526e-05
Time:                        22:05:28   Log-Likelihood:            -1.0559e+05
converged:                       True   LL-Null:                   -1.0559e+05
Covariance Type:            nonrobust   LLR p-value:                    0.1445
===============================================================================================
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      -2.0208      0.029    -70.858      0.000      -2.077      -1.965
C(country)[T.UK]                0.0504      0.029      1.767      0.077      -0.005       0.106
C(country)[T.US]                0.0406      0.027      1.504      0.133      -0.012       0.093
C(time_bucket)[T.15-30_min]    -0.0220      0.016     -1.356      0.175      -0.054       0.010
C(time_bucket)[T.30-45_min]     0.0047      0.016      0.291      0.771      -0.027       0.036
C(time_bucket)[T.45-60_min]    -0.0227      0.016     -1.395      0.163      -0.055       0.009
treat                          -0.0149      0.011     -1.296      0.195      -0.037       0.008
===============================================================================================
"""

## Model 3: Heterogeneous Treatment Effect Model

The third model tests whether the treatment effect differs across countries.

Model:

converted ~ treat * C(country, Treatment(reference='US'))

US is selected as the reference category because it has the largest sample size. This makes the interaction terms easier to interpret.

The interaction terms answer whether the treatment effect in CA or UK differs from the treatment effect in US.

In [15]:
model_interaction = smf.logit(
    "converted ~ treat * C(country, Treatment(reference='US'))",
    data=df
).fit()

model_interaction.summary()

Optimization terminated successfully.
         Current function value: 0.365931
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:              converted   No. Observations:               288541
Model:                          Logit   Df Residuals:                   288535
Method:                           MLE   Df Model:                            5
Date:                Sun, 17 May 2026   Pseudo R-squ.:               3.445e-05
Time:                        22:05:30   Log-Likelihood:            -1.0559e+05
converged:                       True   LL-Null:                   -1.0559e+05
Covariance Type:            nonrobust   LLR p-value:                    0.2009
=====================================================================================================================
                                                        coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------
Intercept                                            -1.9875      0.010   -205.631      0.000      -2.006      -1.969
C(country, Treatment(reference='US'))[T.CA]          -0.0163      0.038     -0.431      0.666      -0.090       0.058
C(country, Treatment(reference='US'))[T.UK]          -0.0054      0.019     -0.288      0.773      -0.042       0.032
treat                                                -0.0201      0.014     -1.468      0.142      -0.047       0.007
treat:C(country, Treatment(reference='US'))[T.CA]    -0.0488      0.054     -0.904      0.366      -0.155       0.057
treat:C(country, Treatment(reference='US'))[T.UK]     0.0306      0.027      1.145      0.252      -0.022       0.083
=====================================================================================================================
"""

## Odds Ratio Tables

Logistic Regression coefficients are expressed in log-odds, which are difficult to interpret directly.

To make interpretation easier, we convert coefficients into odds ratios:

Odds Ratio = exp(coefficient)

Interpretation:

- Odds Ratio > 1: the variable increases conversion odds
- Odds Ratio < 1: the variable decreases conversion odds
- Odds Ratio ≈ 1: the variable has little effect

In [16]:
def create_odds_ratio_table(model, model_name):
    params = model.params
    conf = model.conf_int()
    pvalues = model.pvalues

    odds_table = pd.DataFrame({
        "model": model_name,
        "term": params.index,
        "coefficient": params.values,
        "odds_ratio": np.exp(params.values),
        "ci_lower": np.exp(conf[0].values),
        "ci_upper": np.exp(conf[1].values),
        "p_value": pvalues.values
    })

    return odds_table

## Create odds table for each table

In [17]:
odds_basic = create_odds_ratio_table(
    model_basic,
    "basic_treatment_only"
)

odds_country = create_odds_ratio_table(
    model_country,
    "country_adjusted"
)

odds_tables = [
    odds_basic,
    odds_country
]

if "time_bucket" in df.columns:
    odds_time_adjusted = create_odds_ratio_table(
        model_time_adjusted,
        "country_time_adjusted"
    )
    odds_tables.append(odds_time_adjusted)

odds_interaction = create_odds_ratio_table(
    model_interaction,
    "country_interaction"
)

odds_tables.append(odds_interaction)

logistic_odds_ratios = pd.concat(
    odds_tables,
    ignore_index=True
)

logistic_odds_ratios

,model,term,coefficient,odds_ratio,ci_lower,ci_upper,p_value
0,basic_treatment_only,Intercept,-1.989683,0.136739,0.134586,0.138925,0.000000
1,basic_treatment_only,treat,-0.014863,0.985247,0.963329,1.007663,0.195350
2,country_adjusted,Intercept,-2.030692,0.131245,0.124546,0.138303,0.000000
3,country_adjusted,C(country)[T.UK],0.050311,1.051598,0.994461,1.112018,0.077548
4,country_adjusted,C(country)[T.US],0.040526,1.041358,0.987708,1.097923,0.133185
5,country_adjusted,treat,-0.014814,0.985295,0.963376,1.007712,0.196831
6,country_time_adjusted,Intercept,-2.020752,0.132556,0.125350,0.140176,0.000000
7,country_time_adjusted,C(country)[T.UK],0.050371,1.051661,0.994521,1.112085,0.077195
8,country_time_adjusted,C(country)[T.US],0.040582,1.041417,0.987763,1.097985,0.132652
9,country_time_adjusted,C(time_bucket)[T.15-30_min],-0.022042,0.978199,0.947526,1.009866,0.175101


## Model Comparison

We compare Logistic Regression models using:

- Log-likelihood
- AIC
- BIC
- McFadden pseudo R²

Lower AIC/BIC values generally indicate a better trade-off between model fit and model complexity.

Pseudo R² is not the same as linear regression R². In conversion behavior data, pseudo R² is often small because conversion is influenced by many factors not available in the dataset.

In [18]:
comparison_rows = [
    {
        "model": "basic_treatment_only",
        "log_likelihood": model_basic.llf,
        "aic": model_basic.aic,
        "bic": model_basic.bic,
        "pseudo_r2": model_basic.prsquared
    },
    {
        "model": "country_adjusted",
        "log_likelihood": model_country.llf,
        "aic": model_country.aic,
        "bic": model_country.bic,
        "pseudo_r2": model_country.prsquared
    },
    {
        "model": "country_interaction",
        "log_likelihood": model_interaction.llf,
        "aic": model_interaction.aic,
        "bic": model_interaction.bic,
        "pseudo_r2": model_interaction.prsquared
    }
]

if "time_bucket" in df.columns:
    comparison_rows.insert(
        2,
        {
            "model": "country_time_adjusted",
            "log_likelihood": model_time_adjusted.llf,
            "aic": model_time_adjusted.aic,
            "bic": model_time_adjusted.bic,
            "pseudo_r2": model_time_adjusted.prsquared
        }
    )

model_comparison = pd.DataFrame(comparison_rows)

model_comparison

,model,log_likelihood,aic,bic,pseudo_r2
0,basic_treatment_only,-105588.968900,211181.937800,211203.082985,0.000008
1,country_adjusted,-105587.390028,211182.780056,211225.070426,0.000023
2,country_time_adjusted,-105585.027941,211184.055882,211258.064030,0.000045
3,country_interaction,-105586.169284,211184.338568,211247.774123,0.000034


## Save Logistic Regression Results

The Logistic Regression results are saved as CSV files.

The saved outputs include:

- Model-specific odds ratio tables
- A combined odds ratio table
- Model comparison metrics

In [19]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

odds_basic.to_csv(
    RESULTS_DIR / "logistic_model_basic.csv",
    index=False
)

odds_country.to_csv(
    RESULTS_DIR / "logistic_model_country.csv",
    index=False
)

if "time_bucket" in df.columns:
    odds_time_adjusted.to_csv(
        RESULTS_DIR / "logistic_model_time_adjusted.csv",
        index=False
    )

odds_interaction.to_csv(
    RESULTS_DIR / "logistic_model_interaction.csv",
    index=False
)

logistic_odds_ratios.to_csv(
    RESULTS_DIR / "logistic_odds_ratios.csv",
    index=False
)

model_comparison.to_csv(
    RESULTS_DIR / "logistic_model_comparison.csv",
    index=False
)

print((RESULTS_DIR / "logistic_model_basic.csv").exists())
print((RESULTS_DIR / "logistic_model_country.csv").exists())
print((RESULTS_DIR / "logistic_model_interaction.csv").exists())
print((RESULTS_DIR / "logistic_odds_ratios.csv").exists())
print((RESULTS_DIR / "logistic_model_comparison.csv").exists())

if "time_bucket" in df.columns:
    print((RESULTS_DIR / "logistic_model_time_adjusted.csv").exists())

True
True
True
True
True
True


## Interpretation

Logistic Regression was used as a complementary inference model, not as a pure prediction model.

The baseline model checks the overall treatment effect and is expected to align with the two-proportion z-test.

The country-adjusted model tests whether the treatment effect remains after controlling for country-level baseline differences.

The time-adjusted model checks robustness after controlling for time_bucket.

The interaction model tests whether the treatment effect varies across countries, which corresponds to Heterogeneous Treatment Effect analysis.

The key outputs are:

- treatment coefficient
- treatment p-value
- treatment odds ratio
- interaction p-values
- model comparison metrics

If the treatment odds ratio is below 1 but the p-value is not significant, it suggests that the new page has a slightly negative association with conversion, but the evidence is not strong enough to conclude a statistically significant effect.

## Model Comparison

The fitted logistic regression models are compared using log-likelihood, AIC, BIC, and McFadden pseudo R².

AIC and BIC are used to compare model fit while accounting for model complexity. Lower values generally indicate a better trade-off between fit and complexity.

Pseudo R² is reported as a logistic regression diagnostic, but it should not be interpreted the same way as R² in linear regression.

In [20]:
# =========================
# Model Comparison
# =========================

comparison_rows = [
    {
        "model": "basic_treatment_only",
        "formula": "converted ~ treat",
        "log_likelihood": model_basic.llf,
        "aic": model_basic.aic,
        "bic": model_basic.bic,
        "pseudo_r2": model_basic.prsquared,
        "llr_pvalue": model_basic.llr_pvalue
    },
    {
        "model": "country_adjusted",
        "formula": "converted ~ treat + C(country)",
        "log_likelihood": model_country.llf,
        "aic": model_country.aic,
        "bic": model_country.bic,
        "pseudo_r2": model_country.prsquared,
        "llr_pvalue": model_country.llr_pvalue
    },
    {
        "model": "country_interaction",
        "formula": "converted ~ treat * C(country, Treatment(reference='US'))",
        "log_likelihood": model_interaction.llf,
        "aic": model_interaction.aic,
        "bic": model_interaction.bic,
        "pseudo_r2": model_interaction.prsquared,
        "llr_pvalue": model_interaction.llr_pvalue
    }
]

# Add time-adjusted model only if it exists
if "model_time_adjusted" in globals():
    comparison_rows.insert(
        2,
        {
            "model": "country_time_adjusted",
            "formula": "converted ~ treat + C(country) + C(time_bucket)",
            "log_likelihood": model_time_adjusted.llf,
            "aic": model_time_adjusted.aic,
            "bic": model_time_adjusted.bic,
            "pseudo_r2": model_time_adjusted.prsquared,
            "llr_pvalue": model_time_adjusted.llr_pvalue
        }
    )

model_comparison = pd.DataFrame(comparison_rows)

model_comparison

,model,formula,log_likelihood,aic,bic,pseudo_r2,llr_pvalue
0,basic_treatment_only,converted ~ treat,-105588.968900,211181.937800,211203.082985,0.000008,0.195343
1,country_adjusted,converted ~ treat + C(country),-105587.390028,211182.780056,211225.070426,0.000023,0.184317
2,country_time_adjusted,converted ~ treat + C(country) + C(time_bucket),-105585.027941,211184.055882,211258.064030,0.000045,0.144505
3,country_interaction,"converted ~ treat * C(country, Treatment(refer...",-105586.169284,211184.338568,211247.774123,0.000034,0.200904


## Multicollinearity Check using VIF

Variance Inflation Factor (VIF) is used to check whether explanatory variables are highly correlated with each other.

High multicollinearity can make coefficient estimates unstable and difficult to interpret.

As a rule of thumb:

- VIF < 5: acceptable
- VIF between 5 and 10: moderate concern
- VIF > 10: potential multicollinearity issue

For interaction models, higher VIF values can occur because interaction terms are derived from main-effect variables.

In [21]:
from patsy import dmatrix
from statsmodels.stats.outliers_influence import variance_inflation_factor

# =========================
# VIF Check for Country + Time-Bucket Adjusted Model
# =========================

# Build design matrix for explanatory variables
# We exclude the outcome variable converted
if "time_bucket" in df.columns:
    X_vif = dmatrix(
        "treat + C(country) + C(time_bucket)",
        data=df,
        return_type="dataframe"
    )
else:
    X_vif = dmatrix(
        "treat + C(country)",
        data=df,
        return_type="dataframe"
    )

# Remove intercept for VIF calculation
X_vif_no_intercept = X_vif.drop(columns=["Intercept"], errors="ignore")

# Calculate VIF for each variable
vif_results = pd.DataFrame({
    "variable": X_vif_no_intercept.columns,
    "vif": [
        variance_inflation_factor(X_vif_no_intercept.values, i)
        for i in range(X_vif_no_intercept.shape[1])
    ]
})

vif_results

,variable,vif
0,C(country)[T.UK],1.835248
1,C(country)[T.US],3.349798
2,C(time_bucket)[T.15-30_min],1.839528
3,C(time_bucket)[T.30-45_min],1.838377
4,C(time_bucket)[T.45-60_min],1.838678
5,treat,1.915676


In [22]:
# =========================
# VIF Check for Interaction Model
# =========================

X_interaction_vif = dmatrix(
    "treat * C(country, Treatment(reference='US'))",
    data=df,
    return_type="dataframe"
)

X_interaction_vif_no_intercept = X_interaction_vif.drop(
    columns=["Intercept"],
    errors="ignore"
)

vif_interaction_results = pd.DataFrame({
    "variable": X_interaction_vif_no_intercept.columns,
    "vif": [
        variance_inflation_factor(X_interaction_vif_no_intercept.values, i)
        for i in range(X_interaction_vif_no_intercept.shape[1])
    ]
})

vif_interaction_results

,variable,vif
0,"C(country, Treatment(reference='US'))[T.CA]",2.016531
1,"C(country, Treatment(reference='US'))[T.UK]",1.993380
2,treat,1.426066
3,"treat:C(country, Treatment(reference='US'))[T.CA]",2.088232
4,"treat:C(country, Treatment(reference='US'))[T.UK]",2.347744


In [23]:
def interpret_vif(vif_value):
    if vif_value < 5:
        return "Acceptable"
    elif vif_value < 10:
        return "Moderate concern"
    else:
        return "High multicollinearity concern"

vif_results["interpretation"] = vif_results["vif"].apply(interpret_vif)

vif_results

,variable,vif,interpretation
0,C(country)[T.UK],1.835248,Acceptable
1,C(country)[T.US],3.349798,Acceptable
2,C(time_bucket)[T.15-30_min],1.839528,Acceptable
3,C(time_bucket)[T.30-45_min],1.838377,Acceptable
4,C(time_bucket)[T.45-60_min],1.838678,Acceptable
5,treat,1.915676,Acceptable


In [24]:
vif_interaction_results["interpretation"] = vif_interaction_results["vif"].apply(
    interpret_vif
)

vif_interaction_results

,variable,vif,interpretation
0,"C(country, Treatment(reference='US'))[T.CA]",2.016531,Acceptable
1,"C(country, Treatment(reference='US'))[T.UK]",1.993380,Acceptable
2,treat,1.426066,Acceptable
3,"treat:C(country, Treatment(reference='US'))[T.CA]",2.088232,Acceptable
4,"treat:C(country, Treatment(reference='US'))[T.UK]",2.347744,Acceptable


In [25]:
# =========================
# Save Logistic Regression Results
# =========================

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save model-specific odds ratio tables
odds_basic.to_csv(
    LOGISTIC_MODEL_BASIC_PATH,
    index=False
)

odds_country.to_csv(
    LOGISTIC_MODEL_COUNTRY_PATH,
    index=False
)

# Save time-adjusted model if available
if "odds_time_adjusted" in globals():
    odds_time_adjusted.to_csv(
        LOGISTIC_MODEL_TIME_ADJUSTED_PATH,
        index=False
    )

odds_interaction.to_csv(
    LOGISTIC_MODEL_INTERACTION_PATH,
    index=False
)

# Save combined odds ratio table
logistic_odds_ratios.to_csv(
    LOGISTIC_ODDS_RATIOS_PATH,
    index=False
)

# Save model comparison table
model_comparison.to_csv(
    LOGISTIC_MODEL_COMPARISON_PATH,
    index=False
)

# Save VIF results
vif_results.to_csv(
    LOGISTIC_VIF_RESULTS_PATH,
    index=False
)

# Save interaction VIF results if available
if "vif_interaction_results" in globals():
    vif_interaction_results.to_csv(
        LOGISTIC_VIF_INTERACTION_PATH,
        index=False
    )

# Check saved files
print("Saved logistic_model_basic:", LOGISTIC_MODEL_BASIC_PATH.exists())
print("Saved logistic_model_country:", LOGISTIC_MODEL_COUNTRY_PATH.exists())

if "odds_time_adjusted" in globals():
    print("Saved logistic_model_time_adjusted:", LOGISTIC_MODEL_TIME_ADJUSTED_PATH.exists())

print("Saved logistic_model_interaction:", LOGISTIC_MODEL_INTERACTION_PATH.exists())
print("Saved logistic_odds_ratios:", LOGISTIC_ODDS_RATIOS_PATH.exists())
print("Saved logistic_model_comparison:", LOGISTIC_MODEL_COMPARISON_PATH.exists())
print("Saved logistic_vif_results:", LOGISTIC_VIF_RESULTS_PATH.exists())

if "vif_interaction_results" in globals():
    print("Saved logistic_vif_interaction_results:", LOGISTIC_VIF_INTERACTION_PATH.exists())

Saved logistic_model_basic: True
Saved logistic_model_country: True
Saved logistic_model_time_adjusted: True
Saved logistic_model_interaction: True
Saved logistic_odds_ratios: True
Saved logistic_model_comparison: True
Saved logistic_vif_results: True
Saved logistic_vif_interaction_results: True
